# Join datasets
- Author: Bryan Bravo
- Created: 2026-03-28
## Import Libraries

In [1]:
import os
import sys

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
# in_path = 's3a://ml-project-s3-bronze/input_folder/'
in_path = 'processed_datasets/'


## Custom Functions

## Query


In [4]:
# Read and display the first few rows of each dataset to verify they are loaded correctly.
df_list = ['acled', 'cpi', 'fred', 'gpr', 'lsci', 'oil', 'wb']
for df_name in df_list:
    globals()[f"{df_name}_df"] = spark.read.csv(f"{in_path}/{df_name}.csv", header=True, inferSchema=True)
    print(f"DataFrame: {df_name}")
    globals()[f"{df_name}_df"].show(5)


DataFrame: acled
+---------+-----+----+------+
|  country|month|year|events|
+---------+-----+----+------+
|australia|    1|2021|     2|
|australia|    2|2021|     0|
|australia|    3|2021|     0|
|australia|    4|2021|     0|
|australia|    5|2021|     1|
+---------+-----+----+------+
only showing top 5 rows

DataFrame: cpi
+---------+-----+----+-----+
|  country|value|year|month|
+---------+-----+----+-----+
|australia| 96.4|2024|    4|
|australia|96.17|2024|    5|
|australia|96.52|2024|    6|
|australia|96.77|2024|    7|
|australia|96.49|2024|    8|
+---------+-----+----+-----+
only showing top 5 rows

DataFrame: fred
+--------+---------+-------------+-------+
|    date|  country|interest_rate|fx_rate|
+--------+---------+-------------+-------+
|20060403|australia|          5.5| 0.7177|
|20060404|australia|          5.5| 0.7203|
|20060405|australia|          5.5| 0.7263|
|20060406|australia|          5.5| 0.7315|
|20060407|australia|          5.5| 0.7279|
+--------+---------+-------

In [5]:
# Join acled and cpi
joined_df1 = (
    acled_df.join(cpi_df.withColumnRenamed('value', 'cpi'), on=['year', 'month', 'country'], how='outer')
    .withColumn('events', F.coalesce(F.col('events'), F.lit(0)).cast('int'))  # 0 for missing events, meaning no events reported.
    )

# Join the result with fred
fred_df = (  # extract month and year from date
    fred_df
    .withColumn('date', F.to_date(F.col('date').cast('int'), "yyyyMMdd"))
    .withColumns({
        'year': F.year('date').cast('int'),
        'month': F.month('date').cast('int')
    }) 
)
joined_df2 = (
    joined_df1.join(fred_df, on=['year', 'month', 'country'], how='outer')
)

# Join result with gpr
joined_df3 = (
    joined_df2.join(gpr_df, on=['year', 'month', 'country'], how='outer')
)

# Join result with lsci
joined_df4 = (
    joined_df3.join(lsci_df, on=['year', 'month', 'country'], how='outer')
)

    # .filter(F.col('date') > F.lit('2006-03-31'))  # Filtering early dates to focus on the period of interest 

# Join result with oil
oil_df = (
    oil_df
    .withColumn('date', F.to_date(F.col('date').cast('int'), "yyyyMMdd"))
    .withColumns({
        'year': F.year('date').cast('int'),
        'month': F.month('date').cast('int')
    }).drop('date')
)
joined_df5 = (
    joined_df4.join(oil_df, on=['year', 'month'], how='outer')
)

# Join result with wb
joined_df6 = (
    joined_df5.join(wb_df, on=['year', 'month', 'country'], how='outer')
)

# Repartition, cache, and count the final joined DataFrame to trigger the computation and check the result.
joined_df6.repartition(10).cache().count()


2428853

In [7]:
print(f"Joined DataFrame (row, col) count: {(joined_df6.count(), len(joined_df6.columns))}")
print(joined_df6.toPandas().info(memory_usage='deep'))

Joined DataFrame (row, col) count: (2428853, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2428853 entries, 0 to 2428852
Data columns (total 14 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   year                      int32  
 1   month                     int32  
 2   country                   object 
 3   events                    float64
 4   cpi                       float64
 5   date                      object 
 6   interest_rate             float64
 7   fx_rate                   float64
 8   gpr_index                 float64
 9   lsci                      float64
 10  brent_dollars_per_barrel  float64
 11  wti_dollars_per_barrel    float64
 12  fx_reserves               float64
 13  imports_good_service      float64
dtypes: float64(10), int32(2), object(2)
memory usage: 447.3 MB
None
